# KATL Station Stacking V20 Peak Timing

Single-arm Wunderground-only experiment using V11 Settlement Fix temperature alignment, curated live-safe HRRR/NBM peak-timing features, a 3% train-fold missingness gate, and expanding 2021–2025 validation folds. The readiness section can run while shards are still being pulled and blocks model tuning until coverage is sufficient.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KATL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_v20_peak_timing_stack"
EXPORT_MODEL_WEIGHTS = False
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V20_ENGINEERED_FEATURE_COLUMNS,
    V20_PEAK_TIMING_RAW_FEATURE_COLUMNS,
    build_station_wide_dataset,
    v20_peak_timing_readiness,
    V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS,
    _fit_feature_columns,
    _modeling_frame,
    V11_DROPPED_FEATURE_COLUMNS,
    V11_FEATURE_COLUMNS,
    V20_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V11 Contract

`feature_version="v20_peak_timing"` keeps the v9 feature contract and remaining-warmup target, but trains base learners with Huber-style objectives while retaining the ridge stack selected by validation MAE.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in V20_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_to_2022,2021,2021,2022
1,fold_2021_2022_to_2023,2021,2022,2023
2,fold_2021_2023_to_2024,2021,2023,2024
3,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V11_FEATURE_COLUMNS, sorted(V11_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
0,KATL,gfs,1999,2021-01-01,2026-07-12
1,KATL,hrrr,1998,2021-01-01,2026-06-21
2,KATL,nbm,1998,2021-01-01,2026-07-12


## Model Scores


## Peak-Timing and Wunderground Readiness Gate


In [6]:
readiness_features = build_station_wide_dataset(
    PROJECT_ROOT,
    station_id=STATION_ID,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    feature_version="v20_peak_timing",
    target_source="wunderground_only",
)
EVALUATION_END_DATE = "2026-07-01"  # provisional; official V20 cutoff remains 2026-07-14
v20_ready, readiness_summary, readiness_missing_dates, readiness_fold_missingness = v20_peak_timing_readiness(
    readiness_features,
    station_id=STATION_ID,
    folds=V20_EXPANDING_FOLDS,
    max_missing_fraction=0.03,
    end_date=EVALUATION_END_DATE,
)
readiness_dir = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v20_peak_timing"
readiness_dir.mkdir(parents=True, exist_ok=True)
readiness_summary.to_csv(readiness_dir / f"{STATION_ID}_readiness_summary.csv", index=False)
readiness_missing_dates.to_csv(readiness_dir / f"{STATION_ID}_readiness_missing_dates.csv", index=False)
readiness_fold_missingness.to_csv(readiness_dir / f"{STATION_ID}_readiness_fold_feature_missingness.csv", index=False)
display(readiness_summary)
display(readiness_fold_missingness.groupby(["fold", "retained"], as_index=False).agg(feature_count=("feature", "nunique")))
if not v20_ready:
    raise RuntimeError(
        "V20 readiness failed. Audit artifacts were written; rerun this notebook after peak-timing and "
        "Wunderground pulls reduce each station-year missing fraction to 3% or less."
    )


D:\dev\weather-research\src\calibration\station_stacking.py:3399: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_abs_diff_f"] = (left_values - right_values).abs()
D:\dev\weather-research\src\calibration\station_stacking.py:3398: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_diff_f"] = left_values - right_values
D:\dev\weather-research\src\calibration\station_stacking.py:3399: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

,station_id,year,expected_days,peak_ready_days,wunderground_target_days,peak_missing_fraction,target_missing_fraction,peak_ready,target_ready
0,KATL,2021,365,364,365,0.00274,0.000000,True,True
1,KATL,2022,365,365,364,0.00000,0.002740,True,True
2,KATL,2023,365,365,364,0.00000,0.002740,True,True
3,KATL,2024,366,366,365,0.00000,0.002732,True,True
4,KATL,2025,365,365,364,0.00000,0.002740,True,True
5,KATL,2026,182,182,181,0.00000,0.005495,True,True


,fold,retained,feature_count
0,fold_2021_2022_to_2023,False,1
1,fold_2021_2022_to_2023,True,54
2,fold_2021_2023_to_2024,False,1
3,fold_2021_2023_to_2024,True,54
4,fold_2021_2024_to_2025,False,1
5,fold_2021_2024_to_2025,True,54
6,fold_2021_to_2022,False,1
7,fold_2021_to_2022,True,54
8,test_refit_2021_2025,False,1
9,test_refit_2021_2025,True,54


In [7]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v20_peak_timing",
    training_profile="v20_aligned",
    target_mode="remaining_warmup",
    target_source="wunderground_only",
    max_feature_missing_fraction=0.03,
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=V20_EXPANDING_FOLDS,
    year_split_validation_weights={2022: 1.0, 2023: 1.0, 2024: 1.0, 2025: 1.0},
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v20_peak_timing",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v20_peak_timing/KATL_optuna.sqlite3')

In [8]:
result = run_station_year_split_experiment(config)
result.scoreboard


D:\dev\weather-research\src\calibration\station_stacking.py:3399: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_abs_diff_f"] = (left_values - right_values).abs()
D:\dev\weather-research\src\calibration\station_stacking.py:3398: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_diff_f"] = left_values - right_values
D:\dev\weather-research\src\calibration\station_stacking.py:3399: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,1456,1.409440,1.889919
1,validation_2024_2025,lightgbm,1456,1.324776,1.766115
2,validation_2024_2025,catboost,1456,1.355495,1.808609
3,validation_2024_2025,provider_mean,1456,2.301108,3.339441
4,validation_2024_2025,provider_median,1456,2.143877,3.140707
5,validation_2024_2025,nbm_raw,1456,2.071742,2.972272
6,validation_2024_2025,hrrr_raw,1456,3.100582,4.604907
7,validation_2024_2025,gfs_raw,1456,3.079775,4.204256
8,test_2026,xgboost,171,1.255431,1.721004
9,test_2026,lightgbm,171,1.234041,1.716806


In [9]:
if EXPORT_MODEL_WEIGHTS:
    exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        artifact_dir=config.resolved_output_dir(),
        model_version=MODEL_VERSION,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        training_profile=config.effective_training_profile,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        source_pipeline="notebooks/station_stacking_v20_peak_timing",
    )

    exported_weights.bundle_path, exported_weights.manifest_path
else:
    print("Model export disabled for this experimental notebook.")


Model export disabled for this experimental notebook.


## V11 Feature Coverage


In [10]:
v11_feature_coverage = (
    result.features[V11_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v11_feature_coverage


,feature,coverage_pct
0,v4_observed_precip_any,100.000000
1,v4_forecast_wet_observed_dry,100.000000
2,v4_forecast_observed_precip_match,100.000000
3,v4_all_forecast_precip,100.000000
4,v4_any_forecast_precip,100.000000
5,climatology_high_10y_std_f,100.000000
6,climatology_high_10y_f,100.000000
7,climatology_high_10y_count,100.000000
8,v8_month_remaining_warmup_count,100.000000
9,v4_observed_wet_forecast_dry,100.000000


In [11]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V11_FEATURE_COLUMNS)]


,feature,kind
3,observed_temp_change_last_1h_f,numeric
4,observed_temp_change_last_3h_f,numeric
5,observed_morning_warmup_rate_f_per_hour,numeric
6,observed_high_so_far_change_since_9am_f,numeric
32,v2_recent_heat_anomaly_f,numeric
33,v2_recent_heat_momentum_f,numeric
34,v2_morning_warmup_to_consensus_f,numeric
35,v2_consensus_minus_7d_actual_f,numeric
36,v2_spread_per_warmup_f,numeric
37,v2_humidity_warmup_interaction,numeric


## Dropped Feature Check


In [12]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V11_DROPPED_FEATURE_COLUMNS)
]

dropped_present


,feature,kind


## Morning Trend Coverage


In [13]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,98.812469
1,observed_temp_change_last_3h_f,98.812469
2,observed_morning_warmup_rate_f_per_hour,98.812469
3,observed_high_so_far_change_since_9am_f,98.812469


## Rounded Within 1F Accuracy


In [14]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
7,oof_2026,ridge_stack,171,121,70.760234
8,oof_2026,xgboost,171,120,70.175439
3,oof_2026,lightgbm,171,117,68.421053
0,oof_2026,catboost,171,115,67.251462
4,oof_2026,nbm_raw,171,91,53.216374
6,oof_2026,provider_median,171,84,49.122807
5,oof_2026,provider_mean,171,78,45.614035
2,oof_2026,hrrr_raw,171,71,41.520468
1,oof_2026,gfs_raw,171,59,34.502924
12,validation_2024_2025,lightgbm,1456,962,66.071429


## Version Comparison


In [15]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
    ("v9", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9"),
    ("v10", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10"),
    ("v11", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,lightgbm,137,1.529121,2.224566,v11
1,test_2026,ridge_stack,137,1.531900,2.209782,v11
2,test_2026,lightgbm,125,1.547989,2.093915,v5
3,test_2026,ridge_stack,125,1.566708,2.118374,v5
4,test_2026,xgboost,125,1.569277,2.117086,v5
...,...,...,...,...,...,...
111,validation_2024_2025,gfs_raw,541,3.557627,5.060559,v1
112,validation_2024_2025,hrrr_raw,664,3.630969,5.083107,v2
113,validation_2024_2025,hrrr_raw,664,3.630969,5.083107,v3
114,validation_2024_2025,hrrr_raw,664,3.630969,5.083107,v4


## 2026 OOF Weather Brackets


In [16]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bucket_log_loss,bracket_accuracy_pct,p95_absolute_error_f,large_miss_5f_pct
0,xgboost,171,1.255431,1.721004,1.326794,45.02924,3.428870,1.754386
1,lightgbm,171,1.234041,1.716806,1.324102,44.444444,3.421899,1.754386
2,catboost,171,1.257884,1.715835,1.340251,43.274854,3.363072,1.754386
3,ridge_stack,171,1.227391,1.699855,1.318320,46.783626,3.282840,1.754386
4,provider_mean,171,2.145562,3.337353,1.877072,34.502924,5.357125,6.432749
5,provider_median,171,2.126776,3.276249,1.874809,33.918129,5.905997,7.017544
6,nbm_raw,171,2.088445,3.229287,1.869976,35.672515,5.905997,7.602339
7,hrrr_raw,171,2.539610,3.830303,1.975896,29.824561,7.038774,12.280702
8,gfs_raw,171,2.838918,3.971651,2.083263,21.637427,7.412432,12.865497


## Train-Fold 3% Missingness Audit


In [17]:
modeling_frame, candidate_categorical, candidate_numeric = _modeling_frame(result.features, config)
candidate_features = [*candidate_categorical, *candidate_numeric]
audit_specs = [
    (fold.name, fold.train_start_year, fold.train_end_year)
    for fold in V20_EXPANDING_FOLDS
] + [("test_refit_2021_2025", 2021, 2025)]

missingness_rows = []
years = pd.to_numeric(modeling_frame["year"], errors="coerce")
for fold_name, train_start, train_end in audit_specs:
    train = modeling_frame.loc[years.between(train_start, train_end)].copy()
    retained_categorical, retained_numeric = _fit_feature_columns(
        train,
        candidate_categorical,
        candidate_numeric,
        max_missing_fraction=config.effective_max_feature_missing_fraction,
    )
    retained = set(retained_categorical) | set(retained_numeric)
    for feature in candidate_features:
        numeric_feature = feature in candidate_numeric
        values = pd.to_numeric(train[feature], errors="coerce") if numeric_feature else train[feature]
        missingness_rows.append(
            {
                "fold": fold_name,
                "train_start_year": train_start,
                "train_end_year": train_end,
                "feature": feature,
                "kind": "numeric" if numeric_feature else "categorical",
                "missing_fraction": float(values.isna().mean()),
                "retained": feature in retained,
            }
        )

fold_feature_missingness = pd.DataFrame(missingness_rows)
retained_dropped_summary = (
    fold_feature_missingness.groupby(["fold", "retained"], as_index=False)
    .agg(feature_count=("feature", "nunique"), maximum_missing_fraction=("missing_fraction", "max"))
)
fold_feature_missingness.to_csv(config.resolved_output_dir() / f"{STATION_ID}_fold_feature_missingness.csv", index=False)
retained_dropped_summary, fold_feature_missingness.loc[~fold_feature_missingness["retained"]].sort_values(
    ["fold", "missing_fraction"], ascending=[True, False]
)


(                     fold  retained  feature_count  maximum_missing_fraction
 0  fold_2021_2022_to_2023     False             24                  0.898352
 1  fold_2021_2022_to_2023      True             92                  0.006868
 2  fold_2021_2023_to_2024     False             22                  0.890009
 3  fold_2021_2023_to_2024      True             94                  0.021998
 4  fold_2021_2024_to_2025     False             22                  0.892857
 5  fold_2021_2024_to_2025      True             94                  0.016484
 6       fold_2021_to_2022     False             24                  0.898352
 7       fold_2021_to_2022      True             92                  0.013736
 8    test_refit_2021_2025     False             22                  0.895604
 9    test_refit_2021_2025      True             94                  0.013187,
                        fold  train_start_year  train_end_year  \
 117  fold_2021_2022_to_2023              2021            2022   
 190  fol

## Expanded 11 AM Feature Coverage and Provider Count


In [18]:
new_feature_coverage = (
    result.features[V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)
provider_count_coverage = (
    result.features["v11sf_forecast_temp_11am_provider_count"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("available_provider_count")
    .reset_index(name="row_count")
)
provider_count_coverage["row_pct"] = provider_count_coverage["row_count"] / len(result.features) * 100
new_feature_coverage.to_csv(config.resolved_output_dir() / f"{STATION_ID}_11am_feature_coverage.csv", index=False)
new_feature_coverage, provider_count_coverage


(                                              feature  coverage_pct
 0                     v11sf_forecast_temp_11am_mean_f      98.91143
 1                   v11sf_forecast_temp_11am_median_f      98.91143
 2           v11sf_forecast_temp_11am_minus_observed_f      98.91143
 3                v11sf_forecast_temp_11am_abs_error_f      98.91143
 4               v11sf_forecast_temp_11am_warm_error_f      98.91143
 5               v11sf_forecast_temp_11am_cool_error_f      98.91143
 6                   v11sf_forecast_temp_11am_spread_f      98.91143
 7             v11sf_forecast_temp_11am_provider_count     100.00000
 8   v11sf_forecast_temp_bias_remaining_warmup_inte...      98.91143
 9          v11sf_observation_adjusted_provider_high_f      98.91143
 10                 v11sf_forecast_warmup_after_11am_f      98.91143,
    available_provider_count  row_count    row_pct
 0                         0         22   1.088570
 1                         1         70   3.463632
 2                

## New-Feature Importance


In [19]:
new_feature_importance = result.feature_importance.loc[
    result.feature_importance["feature"].isin(V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS)
].sort_values(["method", "importance_mean_mae_f"], ascending=[True, False])
new_feature_importance


,method,param_key,feature,importance_mean_mae_f,importance_std_mae_f,n_repeats,train_start_year,train_end_year,test_year,train_rows,test_rows
78,catboost,trial_27,v11sf_forecast_temp_bias_remaining_warmup_inte...,0.006658,0.002734,10,2021,2025,2026,1820,171
97,catboost,trial_27,v11sf_forecast_warmup_after_11am_f,0.004572,0.011833,10,2021,2025,2026,1820,171
106,catboost,trial_27,v11sf_forecast_temp_11am_warm_error_f,0.003948,0.001109,10,2021,2025,2026,1820,171
108,catboost,trial_27,v11sf_forecast_temp_11am_minus_observed_f,0.003886,0.002122,10,2021,2025,2026,1820,171
119,catboost,trial_27,v11sf_forecast_temp_11am_abs_error_f,0.002994,0.001548,10,2021,2025,2026,1820,171
121,catboost,trial_27,v11sf_forecast_temp_11am_mean_f,0.002811,0.002732,10,2021,2025,2026,1820,171
133,catboost,trial_27,v11sf_observation_adjusted_provider_high_f,0.002120,0.001812,10,2021,2025,2026,1820,171
158,catboost,trial_27,v11sf_forecast_temp_11am_cool_error_f,0.000760,0.001538,10,2021,2025,2026,1820,171
161,catboost,trial_27,v11sf_forecast_temp_11am_spread_f,0.000702,0.002069,10,2021,2025,2026,1820,171
192,catboost,trial_27,v11sf_forecast_temp_11am_provider_count,0.000000,0.000000,10,2021,2025,2026,1820,171


## 2026 Monthly Metrics


In [20]:
monthly_predictions = result.test_predictions.copy()
monthly_predictions["month"] = pd.to_datetime(monthly_predictions["contract_date"], errors="coerce").dt.month
monthly_metrics = (
    monthly_predictions.dropna(subset=["month", "error_f"])
    .groupby(["method", "month"], as_index=False)
    .agg(
        count=("error_f", "size"),
        mae_f=("absolute_error_f", "mean"),
        rmse_f=("error_f", lambda values: float(np.sqrt(np.mean(np.square(values))))),
        bias_f=("error_f", "mean"),
    )
)
monthly_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_2026_monthly_metrics.csv", index=False)
monthly_metrics


,method,month,count,mae_f,rmse_f,bias_f
0,catboost,1,31,1.203912,1.483021,0.017164
1,catboost,2,28,1.210628,1.482858,0.154812
2,catboost,3,30,1.056130,1.365085,-0.080902
3,catboost,4,30,1.502376,2.108195,-0.810535
4,catboost,5,31,1.611793,2.269823,0.254005
5,catboost,6,21,0.817071,1.085356,-0.263995
6,gfs_raw,1,31,2.802349,3.395373,2.287120
7,gfs_raw,2,28,3.106535,3.827184,2.926251
8,gfs_raw,3,30,3.179913,5.493333,1.720638
9,gfs_raw,4,30,2.083964,2.684767,-0.458210


## Performance by Warm/Cool 11 AM Forecast Delta


In [21]:
delta_by_date = result.features[[
    "contract_date",
    "v11sf_forecast_temp_11am_minus_observed_f",
]].copy()
delta_predictions = result.test_predictions.merge(delta_by_date, on="contract_date", how="left")
delta_predictions["forecast_temp_delta_bucket"] = pd.cut(
    delta_predictions["v11sf_forecast_temp_11am_minus_observed_f"],
    bins=[-np.inf, -2.0, -0.5, 0.5, 2.0, np.inf],
    labels=["cool_gt_2f", "cool_0.5_to_2f", "near_match", "warm_0.5_to_2f", "warm_gt_2f"],
)
warm_cool_metrics = (
    delta_predictions.dropna(subset=["forecast_temp_delta_bucket", "error_f"])
    .groupby(["method", "forecast_temp_delta_bucket"], observed=True, as_index=False)
    .agg(count=("error_f", "size"), mae_f=("absolute_error_f", "mean"), bias_f=("error_f", "mean"))
)
warm_cool_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_warm_cool_delta_metrics.csv", index=False)
warm_cool_metrics


,method,forecast_temp_delta_bucket,count,mae_f,bias_f
0,catboost,cool_gt_2f,34,1.369930,0.211657
1,catboost,cool_0.5_to_2f,54,1.018878,0.013303
2,catboost,near_match,39,1.233332,-0.516788
3,catboost,warm_0.5_to_2f,29,1.270417,-0.320360
4,catboost,warm_gt_2f,15,1.903932,0.132296
5,gfs_raw,cool_gt_2f,34,3.787576,3.739109
6,gfs_raw,cool_0.5_to_2f,54,2.832964,1.668038
7,gfs_raw,near_match,39,2.583485,-0.293216
8,gfs_raw,warm_0.5_to_2f,29,2.390429,-0.388416
9,gfs_raw,warm_gt_2f,15,2.241267,-1.095862


## Common-Date Comparison with Existing V11 Settlement Fix


In [22]:
baseline_path = (
    PROJECT_ROOT
    / "data"
    / "calibration"
    / "station_stacking_v11_settlement_fix"
    / f"{STATION_ID}_year_split_test_predictions.csv"
)
baseline_predictions = pd.read_csv(baseline_path)
baseline_predictions["contract_date"] = baseline_predictions["contract_date"].astype(str).str[:10]
v20_predictions = result.test_predictions.copy()
v20_predictions["contract_date"] = v20_predictions["contract_date"].astype(str).str[:10]
comparison = baseline_predictions.merge(
    v20_predictions,
    on=["contract_date", "method"],
    suffixes=("_baseline", "_v20"),
)
comparison["baseline_abs_error_f"] = (
    pd.to_numeric(comparison["actual_high_f_baseline"], errors="coerce")
    - pd.to_numeric(comparison["predicted_high_f_baseline"], errors="coerce")
).abs()
comparison["v20_abs_error_f"] = (
    pd.to_numeric(comparison["actual_high_f_v20"], errors="coerce")
    - pd.to_numeric(comparison["predicted_high_f_v20"], errors="coerce")
).abs()
common_date_comparison = (
    comparison.groupby("method", as_index=False)
    .agg(
        common_date_count=("contract_date", "size"),
        baseline_mae_f=("baseline_abs_error_f", "mean"),
        v20_mae_f=("v20_abs_error_f", "mean"),
        v20_better_days=("v20_abs_error_f", lambda values: int((values < comparison.loc[values.index, "baseline_abs_error_f"]).sum())),
        baseline_better_days=("v20_abs_error_f", lambda values: int((values > comparison.loc[values.index, "baseline_abs_error_f"]).sum())),
    )
)
common_date_comparison["v20_delta_mae_f"] = common_date_comparison["v20_mae_f"] - common_date_comparison["baseline_mae_f"]
common_date_comparison.to_csv(config.resolved_output_dir() / f"{STATION_ID}_v11_common_date_comparison.csv", index=False)
common_date_comparison.sort_values("v20_delta_mae_f")


,method,common_date_count,baseline_mae_f,v20_mae_f,v20_better_days,baseline_better_days,v20_delta_mae_f
3,lightgbm,171,1.359759,1.234041,99,70,-0.125717
7,ridge_stack,171,1.326830,1.227391,97,74,-0.099439
0,catboost,171,1.352620,1.257884,99,70,-0.094737
8,xgboost,171,1.326999,1.255431,90,78,-0.071567
2,hrrr_raw,171,2.539610,2.539610,0,0,0.000000
4,nbm_raw,171,2.088445,2.088445,0,0,0.000000
5,provider_mean,171,2.145562,2.145562,3,2,0.000000
1,gfs_raw,171,2.838918,2.838918,0,0,0.000000
6,provider_median,171,2.126776,2.126776,0,0,0.000000


## V20 Peak-Timing Feature Coverage


In [23]:
v20_feature_columns = [*V20_PEAK_TIMING_RAW_FEATURE_COLUMNS, *V20_ENGINEERED_FEATURE_COLUMNS]
v20_feature_coverage = (
    result.features[v20_feature_columns]
    .notna().mean().mul(100)
    .rename("coverage_pct").reset_index().rename(columns={"index": "feature"})
)
v20_feature_coverage.to_csv(
    config.resolved_output_dir() / f"{STATION_ID}_v20_peak_feature_coverage.csv", index=False
)
v20_feature_coverage.sort_values("coverage_pct")


,feature,coverage_pct
5,nbm_cooling_onset_hour_local,89.460663
30,v20_nbm_observation_adjusted_high_f,98.861950
26,v20_nbm_t11_minus_observed_f,98.861950
25,v20_hrrr_t11_minus_observed_f,98.911430
29,v20_hrrr_observation_adjusted_high_f,98.911430
31,v20_adjusted_high_mean_f,98.911430
32,v20_adjusted_high_spread_f,98.911430
2,nbm_hour_of_max_local,99.950520
17,hrrr_precip_wet_hours_11_to_nbm_peak,99.950520
22,hrrr_tcc_11_to_nbm_peak_max_pct,99.950520


## Fold Metrics and Peak-Feature Importance


In [24]:
fold_metrics = (
    result.validation_predictions.groupby(["fold", "method"], as_index=False)
    .agg(count=("absolute_error_f", "size"), mae_f=("absolute_error_f", "mean"), bias_f=("error_f", "mean"))
)
peak_feature_importance = result.feature_importance.loc[
    result.feature_importance["feature"].isin(v20_feature_columns)
].sort_values(["method", "importance_mean_mae_f"], ascending=[True, False])
fold_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_fold_metrics.csv", index=False)
fold_metrics, peak_feature_importance


(                      fold           method  count     mae_f    bias_f
 0   fold_2021_2022_to_2023         catboost    363  1.452003 -0.454867
 1   fold_2021_2022_to_2023          gfs_raw    363  2.906738 -0.398481
 2   fold_2021_2022_to_2023         hrrr_raw    363  3.158485  1.943606
 3   fold_2021_2022_to_2023         lightgbm    363  1.451739 -0.393612
 4   fold_2021_2022_to_2023          nbm_raw    363  1.926042  1.063129
 5   fold_2021_2022_to_2023    provider_mean    363  2.153135  0.869418
 6   fold_2021_2022_to_2023  provider_median    363  1.939755  0.699834
 7   fold_2021_2022_to_2023          xgboost    363  1.493802 -0.501828
 8   fold_2021_2023_to_2024         catboost    365  1.226957  0.221697
 9   fold_2021_2023_to_2024          gfs_raw    365  2.692234  0.106500
 10  fold_2021_2023_to_2024         hrrr_raw    365  2.866136  1.852979
 11  fold_2021_2023_to_2024         lightgbm    365  1.202714  0.165045
 12  fold_2021_2023_to_2024          nbm_raw    365  1.837451  1